In [ ]:
#%pip install pandas torch transformers datasets evaluate sacrebleu bert-score sentence-transformers rapidfuzz deep-translator tqdm numpy matplotlib PyYAML

In [ ]:
# Ячейка 1: Импорты и загрузка конфига
import sys
sys.path.append('..')
from utils import load_config, build_dataloaders
import pandas as pd
import importlib

config = load_config("config.yaml")
print("Config loaded:")
print(config)

In [ ]:
# Ячейка 2: Создание датасета
source = config["dataset_source"]
source_params = config["source_params"].get(source, {})
module = importlib.import_module(f"dataset.create_dataset.{source}")
df = getattr(module, "load_or_create")(source_params)
print(f"Dataset size: {len(df)}")
df.head()

In [ ]:
# Ячейка 3: Предобработка
from dataset.prepare_dataset.prepare import prepare_dataset
df_clean = prepare_dataset(df, config["preprocessing"])
print(f"Clean dataset size: {len(df_clean)}")

In [ ]:
# Ячейка 4: Построение даталоадеров
train_loader, val_loader, test_loader = build_dataloaders(df_clean, config["model_config"])
print(f"Train samples: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}")

In [ ]:
# Ячейка 5: Создание модели и обучение
from models.rut5_paraphraser.model import ParaphraserModel
from models.trainer import train_model

model = ParaphraserModel(config["model_config"], device="cuda" if __import__('torch').cuda.is_available() else "cpu")
trained_model = train_model(
    model,
    train_loader,
    val_loader,
    config["model_config"],
    config["trainer_config"],
    config["metrics_config"]
)
trained_model.save(config["trainer_config"]["output_dir"])

In [ ]:
# Ячейка 6: Оценка на тесте
from models.model_use import evaluate_model
test_metrics = evaluate_model(trained_model, test_loader, config["metrics_config"])
print("Test metrics:", test_metrics)